In [1]:
from platform import python_version
print(python_version())

3.11.14


In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, ROOT_SRC)


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config


from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/PAAD/config/all_lfc_cutoffs_PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/PAAD
>>> PAAD Tumor
>>> case Tumor
>>> psi_id or disease: PAAD
Error: No data available for the specified PAAD.
Error: could not find /home/flavio/uv/perturb_agent/data/TCGA/PAAD/lfc/PAAD_final_LFC_Tumor_x_CTRL_not_normalized.tsv
No dflfc table was calculated for this case Tumor

Echo Parameters:
	0/0 DEGs/ensembl.
		Up 0/0 DEGs/ensembl.
		Dw 0/0 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.


### cBioPortal - no memory restriction to get all data available

In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
verbose = True

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

Table opened ((7, 9)) at '/home/flavio/uv/perturb_agent/data/cbioportal_study_mapping.tsv'


,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
0,TCGA,paad_tcga_pan_can_atlas_2018,True,TCGA-PAAD,PAAD,pancreatic_adenocarcinoma,PAAD,Pancreas,"TCGA pancreatic adenocarcinoma, PanCancer Atlas"
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"
2,TCGA,skcm_tcga_pan_can_atlas_2018,True,TCGA-SKCM,SKCM,cutaneous_melanoma,SKCM,Skin,"TCGA skin cutaneous melanoma, PanCancer Atlas"
3,TCGA,brca_tcga_pan_can_atlas_2018,True,TCGA-BRCA,BRCA,breast_invasive_carcinoma,BRCA,Breast,"TCGA breast invasive carcinoma, PanCancer Atlas"
4,CPTAC2,brca_cptac_2020,True,CPTAC-2,BRCA,breast_cancer,BRCA,Breast,"CPTAC breast cancer publication cohort, Cell 2020"


In [7]:
"; ".join(cbio.prog_list)

'CPTAC2; CPTAC3; TCGA'

### Open primary cites from cbio

In [8]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

### Get cases, subtypes and clin_demo tables

In [9]:
verbose = True

cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

df_cases, df_subt, df_clin_demo, df_case_bar = cbio.get_cases_and_subtypes(batch_size=200, verbose=verbose)

df_cases.shape, df_clin_demo.shape, df_case_bar.shape


-----------------------------
>> prog_id: CPTAC3
>> psi_id: PAAD
>> primary_site: Pancreas
>> disease_id: pancreatic_ductal_adenocarcinoma
>> disease_cd: PAAD

-----------------------------
>> cbioportal_study_id: paad_cptac_2021
>> gdc_project_id: CPTAC-3

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations
-----------------------------

Table opened ((170, 27)) at '/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/cases_for_PAAD.tsv'
Table opened ((140, 20)) at '/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/clinical_and_demographics_for_paad_cptac_2021.tsv'


((170, 27), (140, 20), (140, 2))

### Calc expression

 - calc_file_expression_tumor_normal_gtex()
   - get_dic_expression_tumor_and_normal()
     - get_filtered_tables()
     - get_table_given_fileID()
   - prepare_normal_tumor_tables()

  
#### Tables in: root_disease / lfc

In [10]:
cbio.root_disease, cbio.root_lfc, cbio.filename_demo

(PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD'),
 PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc'),
 PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/clinical_and_demographics_for_paad_cptac_2021.tsv'))

In [11]:
df_case_bar.head(3)

,case_id,barcode_case
0,b288852f-66f5-4949-bc13-883e634313fc,C3N-02592
1,45669b98-2d82-41e1-9c1d-0de3f98cebe1,C3L-00277
2,81c1e904-8a47-462c-bd2c-f2ecbb6c1b19,C3L-01637


In [12]:
df_cases.head(3)

,primary_site,disease_type,case_id,barcode_case,diagnoses,gdc_project_id,subtype_global,stage_ajcc,primary_diagnosis,tumor_grade,...,primary_site_norm,disease_type_norm,diagnosis_norm,tumor_class,histology,subtype_tissue,consistency,validity,n,frac
0,Pancreas,Ductal and Lobular Neoplasms,b288852f-66f5-4949-bc13-883e634313fc,C3N-02592,"[{'primary_diagnosis': 'Infiltrating duct carcinoma, NOS'}]",CPTAC-3,other,unknown,"Infiltrating duct carcinoma, NOS",NaN,...,pancreas,ductal and lobular neoplasms,infiltrating duct carcinoma,other,other,other,ok,ambiguous,1,0.006
1,Pancreas,Ductal and Lobular Neoplasms,45669b98-2d82-41e1-9c1d-0de3f98cebe1,C3L-00277,"[{'primary_diagnosis': 'Infiltrating duct carcinoma, NOS'}]",CPTAC-3,other,unknown,"Infiltrating duct carcinoma, NOS",NaN,...,pancreas,ductal and lobular neoplasms,infiltrating duct carcinoma,other,other,other,ok,ambiguous,1,0.006
2,Pancreas,Ductal and Lobular Neoplasms,81c1e904-8a47-462c-bd2c-f2ecbb6c1b19,C3L-01637,"[{'primary_diagnosis': 'Infiltrating duct carcinoma, NOS'}]",CPTAC-3,other,unknown,"Infiltrating duct carcinoma, NOS",NaN,...,pancreas,ductal and lobular neoplasms,infiltrating duct carcinoma,other,other,other,ok,ambiguous,1,0.006


In [13]:
df_clin_demo['barcode_case'] = [x[1:] if x.startswith('X') else x for x in df_clin_demo.barcode_case]

In [14]:
# df_cases[df_cases.barcode_case.isin(df_clin_demo.barcode_case)]

In [15]:
"; ".join(df_clin_demo.barcode_case)

'C3L-00017; C3L-00102; C3L-00189; C3L-00277; C3L-00401; C3L-00589; C3L-00598; C3L-00599; C3L-00622; C3L-00625; C3L-00640; C3L-00819; C3L-00881; C3L-00928; C3L-01031; C3L-01036; C3L-01037; C3L-01051; C3L-01052; C3L-01053; C3L-01054; C3L-01124; C3L-01328; C3L-01453; C3L-01598; C3L-01637; C3L-01662; C3L-01687; C3L-01689; C3L-01703; C3L-01971; C3L-02109; C3L-02112; C3L-02115; C3L-02116; C3L-02118; C3L-02463; C3L-02604; C3L-02606; C3L-02610; C3L-02613; C3L-02701; C3L-02809; C3L-02890; C3L-02897; C3L-02899; C3L-03123; C3L-03356; C3L-03388; C3L-03394; C3L-03395; C3L-03628; C3L-03630; C3L-03632; C3L-03635; C3L-03639; C3L-03743; C3L-04027; C3L-04072; C3L-04080; C3L-04473; C3L-04475; C3L-04479; C3L-04495; C3L-04848; C3L-04853; C3N-00198; C3N-00249; C3N-00302; C3N-00303; C3N-00436; C3N-00511; C3N-00512; C3N-00513; C3N-00514; C3N-00516; C3N-00517; C3N-00518; C3N-00709; C3N-00957; C3N-01011; C3N-01012; C3N-01165; C3N-01166; C3N-01167; C3N-01168; C3N-01375; C3N-01380; C3N-01381; C3N-01382; C3N-01383

In [16]:
df_clin_demo.head(3).T

,0,1,2
barcode_case,C3L-00017,C3L-00102,C3L-00189
age,69,42,68
alcohol_consumption,Alcohol consumption equal to or less than 2 drinks per day for men and 1 dri...,Alcohol consumption history not available,Alcohol consumption equal to or less than 2 drinks per day for men and 1 dri...
bmi,28.36,26.93,34.28
cause_of_death,NaN,Pancreatic Carcinoma,Pancreatic Carcinoma
follow_up_days,426.0,249.0,1035.0
histology_diagnosis,Pancreatic Ductal Adenocarcinoma,Pancreatic Ductal Adenocarcinoma,Pancreatic Ductal Adenocarcinoma
is_this_patient_lost_to_follow_up,Yes,No,No
medical_condition,"Diabetes mellitus type II (NIDDM, adult onset diabetes)|hyperlipidemia|Neck ...","Hypertension|hyperlipidemia|Diabetes mellitus type II (NIDDM, adult onset di...",Tachycardia|Hypertension|sleep apnea|cholecystectomy|jaundice|allergies|Hyst...
participant_country,United States,United States,Canada


### Run a program

In [17]:
verbose=False
force=False

prog_id_list = ['CPTAC', 'CCLE', 'TARGET', '']
prog_id_list = ['TCGA']
prog_id_list = ['CPTAC2']
prog_id_list = ['TCGA', 'CPTAC2', 'CPTAC3']

for i, prog_id in enumerate(prog_id_list):

    print(f"{i}) prog_id {prog_id}")

    df_all_cases, df_all_clin_demo, df_all_case_bar, df_all_samples, df_all_mutations = cbio.loop_program_psi_get_cases_samples_mut(prog_id=prog_id, force=force, verbose=verbose)

    print("")

print("\n--------------- end ---------------")


0) prog_id TCGA

1) prog_id CPTAC2

2) prog_id CPTAC3


--------------- end ---------------


In [18]:
prog_id = 'CPTAC3'
prog_id = 'TCGA'
prog_id = 'CPTAC2'

force=False
verbose=False

df_all_cases, df_all_clin_demo, df_all_case_bar, df_all_samples, df_all_mutations = cbio.loop_program_psi_get_cases_samples_mut(prog_id=prog_id, force=force, verbose=verbose)

len(df_cases), len(df_all_clin_demo), len(df_all_case_bar)

(170, 122, 120)

In [19]:
df_all_clin_demo.head(3)

,barcode_case,age,apobec_signature,cd3_tils_counts,cd3_tils_status,chromosome_instability_index_cin_,cibersort_absolute_score,erbb2_gene_amplified,erbb2_proteogenomic_status,erbb2_updated_clinical_status,...,sample_count,gender,stemness_score,tnbc_updated_clinical_status,top2a_gene_amplified,top2a_proteogenomic_status,xcell_immune_score,xcell_stromal_score,bmi,cbioportal_study_id
0,CPT000814,NaN,N,not performed,not performed,2.272,0.565,0,Negative,NaN,...,1,NaN,0.798,Positive,0,negative,0.041,0.000,NaN,brca_cptac_2020
1,CPT001846,NaN,N,not performed,not performed,0.827,0.786,0,Negative,NaN,...,1,NaN,0.592,Positive,0,negative,0.077,0.024,NaN,brca_cptac_2020
2,01BR001,55.0,N,not performed,not performed,1.138,0.454,0,Negative,Negative,...,1,Female,0.839,Positive,0,negative,0.010,0.009,NaN,brca_cptac_2020


In [20]:
df_all_case_bar.head(3)

,case_id,barcode_case
0,1f6f6c63-1202-4b8c-9b13-f27f9022d556,01BR040
1,c0cef5fc-0c80-4812-aece-97c351747f26,11BR032
2,e2549d98-1b68-4c4d-b7f6-311fdd05b260,14BR008


In [21]:
df_attr_gdc = cbio.check_clinical_attributes(
    "pancreas_cptac_gdc"
)

df_attr_2021 = cbio.check_clinical_attributes(
    "paad_cptac_2021"
)

display(df_attr_gdc)
display(df_attr_2021)

,displayName,description,datatype,patientAttribute,priority,clinicalAttributeId,studyId
0,Diagnosis Age,Age at the time of diagnosis expressed in number of days since birth.,NUMBER,True,1,AGE,pancreas_cptac_gdc
1,Sex,Text designations that identify gender. Gender is described as the assemblag...,STRING,True,1,SEX,pancreas_cptac_gdc


,displayName,description,datatype,patientAttribute,priority,clinicalAttributeId,studyId
0,Age,Age in years,NUMBER,True,1,AGE,paad_cptac_2021
1,BMI,Body mass index,NUMBER,True,1,BMI,paad_cptac_2021
2,Tumor Stage Pathological,Tumor stage pathological,STRING,False,1,TUMOR_STAGE_PATHOLOGICAL,paad_cptac_2021


In [22]:
prog_ids = ['TCGA-KIRP', 'TCGA-THCA', 'CGCI-BLGSP', 'FM-AD', 'TCGA-LAML', 'TCGA-COAD', 
            'CPTAC-2', 'HCMI-CMDC', 'MMRF-COMMPASS', 'RC-PTCL', 'WCDT-MCRPC', 'ALCHEMIST-ALCH', 'MATCH-U', 'APOLLO-LUAD', 'CMI-ASC', 'MATCH-Z1A',
            'CGCI-HTMCP-DLBCL', 'TCGA-MESO', 'TARGET-ALL-P3', 'VAREPOP-APOLLO', 'MATCH-C1', 'MATCH-S2', 'TCGA-PRAD', 'TARGET-CCSK', 'TARGET-AML', 
            'MATCH-Y', 'TCGA-ACC', 'MATCH-R', 'MP2PRT-ALL', 'TARGET-RT', 'MATCH-Z1D', 'CPTAC-3', 'TCGA-LGG', 'TCGA-SARC', 'APOLLO-OV', 'MATCH-B', 
            'TCGA-LUSC', 'BEATAML1.0-COHORT', 'TCGA-ESCA', 'TARGET-ALL-P1', 'TCGA-KICH', 'OHSU-CNL', 'TCGA-LIHC', 'TCGA-TGCT', 'TCGA-OV', 'MATCH-W', 
            'TCGA-GBM', 'EXCEPTIONAL_RESPONDERS-ER', 'TCGA-READ', 'TARGET-OS', 'TRIO-CRU', 'TCGA-THYM', 'TCGA-HNSC', 'TCGA-UCEC', 'TARGET-NBL', 
            'MATCH-S1', 'TARGET-ALL-P2', 'CDDP_EAGLE-1', 'TCGA-SKCM', 'TCGA-PCPG', 'ORGANOID-PANCREATIC', 'MATCH-N', 'CGCI-HTMCP-CC', 'CCG-CUPP',
            'MATCH-I', 'BEATAML1.0-CRENOLANIB', 'CTSP-DLBCL1', 'CGCI-HTMCP-LC', 'TCGA-STAD', 'TCGA-BRCA', 'NCICCR-DLBCL', 'MP2PRT-WT', 'CCDI-MCI',
            'TCGA-PAAD', 'MATCH-Q', 'MATCH-H', 'TCGA-CHOL', 'REBC-THYR', 'MATCH-Z1B', 'CMI-MBC', 'TARGET-WT', 'TCGA-CESC', 'MATCH-P', 'TCGA-UCS', 'CMI-MPC', 
            'TCGA-DLBC', 'TCGA-KIRC', 'TCGA-LUAD', 'MATCH-Z1I', 'TCGA-UVM', 'TCGA-BLCA']

prog_ids.sort()

print("; ".join(prog_ids))

ALCHEMIST-ALCH; APOLLO-LUAD; APOLLO-OV; BEATAML1.0-COHORT; BEATAML1.0-CRENOLANIB; CCDI-MCI; CCG-CUPP; CDDP_EAGLE-1; CGCI-BLGSP; CGCI-HTMCP-CC; CGCI-HTMCP-DLBCL; CGCI-HTMCP-LC; CMI-ASC; CMI-MBC; CMI-MPC; CPTAC-2; CPTAC-3; CTSP-DLBCL1; EXCEPTIONAL_RESPONDERS-ER; FM-AD; HCMI-CMDC; MATCH-B; MATCH-C1; MATCH-H; MATCH-I; MATCH-N; MATCH-P; MATCH-Q; MATCH-R; MATCH-S1; MATCH-S2; MATCH-U; MATCH-W; MATCH-Y; MATCH-Z1A; MATCH-Z1B; MATCH-Z1D; MATCH-Z1I; MMRF-COMMPASS; MP2PRT-ALL; MP2PRT-WT; NCICCR-DLBCL; OHSU-CNL; ORGANOID-PANCREATIC; RC-PTCL; REBC-THYR; TARGET-ALL-P1; TARGET-ALL-P2; TARGET-ALL-P3; TARGET-AML; TARGET-CCSK; TARGET-NBL; TARGET-OS; TARGET-RT; TARGET-WT; TCGA-ACC; TCGA-BLCA; TCGA-BRCA; TCGA-CESC; TCGA-CHOL; TCGA-COAD; TCGA-DLBC; TCGA-ESCA; TCGA-GBM; TCGA-HNSC; TCGA-KICH; TCGA-KIRC; TCGA-KIRP; TCGA-LAML; TCGA-LGG; TCGA-LIHC; TCGA-LUAD; TCGA-LUSC; TCGA-MESO; TCGA-OV; TCGA-PAAD; TCGA-PCPG; TCGA-PRAD; TCGA-READ; TCGA-SARC; TCGA-SKCM; TCGA-STAD; TCGA-TGCT; TCGA-THCA; TCGA-THYM; TCGA-UCEC; TCG